# Thai-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: Thai (th) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture.
We use the **ALT (Asian Language Treebank)** dataset for Thai-English parallel data.
We use **SentencePiece** for subword tokenization.

## Pipeline
1.  **Setup**: Install/Import dependencies.
2.  **Data Loading**: Load the ALT dataset (Thai-English).
3.  **Tokenization**: Train SentencePiece model (`spm_th`, `spm_en_th`).
4.  **Data Processing**: Create PyTorch Datasets and DataLoaders.
5.  **Model**: Implement Transformer.
6.  **Training**: Train the model.
7.  **Evaluation**: Calculate BLEU score.
8.  **Inference**: Demo function and save model for Web App.

## 1. Setup and Imports

In [1]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Using device: cuda


In [2]:
# Install dependencies if missing (uncomment if needed)
# !pip install sentencepiece datasets portalocker

## 2. Data Loading (ALT Dataset)
Loading Thai-English pairs from ALT.

In [3]:
from datasets import load_dataset

print("Loading ALT Dataset (Thai-English)...")
try:
    dataset = load_dataset("alt", split="train+validation+test")
    print(f"Loaded {len(dataset)} sentences from ALT dataset.")

    # Filter/Extract only Thai and English
    data = []
    for item in dataset:
        if 'translation' in item:
            if 'th' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'th': item['translation']['th'],
                    'en': item['translation']['en']
                })

    print(f"Extracted {len(data)} Thai-English pairs.")

except Exception as e:
    print(f"Error loading from HF: {e}")


Loading ALT Dataset (Thai-English)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alt-parallel/train-00000-of-00001.parque(…):   0%|          | 0.00/31.2M [00:00<?, ?B/s]

alt-parallel/validation-00000-of-00001.p(…):   0%|          | 0.00/1.71M [00:00<?, ?B/s]

alt-parallel/test-00000-of-00001.parquet:   0%|          | 0.00/1.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18088 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1019 [00:00<?, ? examples/s]

Loaded 20107 sentences from ALT dataset.
Extracted 20107 Thai-English pairs.


In [4]:
# Convert to DataFrame
df = pd.DataFrame(data)
print(df.head())

# Basic Cleaning
df = df.dropna(subset=['th', 'en'])
df['th'] = df['th'].astype(str)
df['en'] = df['en'].astype(str)

df = df[df['th'].str.strip() != '']
df = df[df['en'].str.strip() != '']
print(f"After cleaning: {len(df)} pairs")

print("\n--- Data Alignment Check ---")
for i in range(5):
    sample = df.sample(1).iloc[0]
    print(f"Source (th): {sample['th']}")
    print(f"Target (en): {sample['en']}")
    print("-" * 20)

                                                  th  \
0  อิตาลีได้เอาชนะโปรตุเกสด้วยคะแนน31ต่อ5 ในกลุ่ม...   
1  Andrea Masi ได้เปิดฉากทำคะแนนในนาทีที่สี่ ด้วย...   
2  ทั้งที่เป็นฝ่ายคุมเกมส์ในครึ่งแรกของการแข่งขัน...   
3  โปรตุเกสไม่ละความพยยาม และDavid Penalvaได้ทำคะ...   
4  ในครึ่งแรกอิตาลีขึ้นนำด้วยคะแนน16 ต่อ5 แต่ถูกป...   

                                                  en  
0  Italy have defeated Portugal 31-5 in Pool C of...  
1  Andrea Masi opened the scoring in the fourth m...  
2  Despite controlling the game for much of the f...  
3  Portugal never gave up and David Penalva score...  
4  Italy led 16-5 at half time but were matched b...  
After cleaning: 20101 pairs

--- Data Alignment Check ---
Source (th): ปัญหาเริ่มจากมีกลุ่ม "นักปีนเขาที่สวมแต่รองเท้าบูทเท่านั้น" ถูกตำรวจที่ Alpine จับเมื่อฤดูใบไม้ร่วงที่ผ่านมา
Target (en): The problem started with a group of "boot-only hikers" who were stopped by the police in the Alpine region last autumn.
--------------------

## 3. Tokenization (SentencePiece)
Training separate tokenizers for Thai (`spm_th`) and English (`spm_en_th`).

In [5]:
import sentencepiece as spm

# 1. Save texts to files
with open('train_th.txt', 'w', encoding='utf-8') as f:
    for line in df['th']:
        f.write(line + '\n')

with open('train_en_th.txt', 'w', encoding='utf-8') as f:
    for line in df['en']:
        f.write(line + '\n')

# 2. Train SentencePiece models
vocab_size = 4000
model_type = 'bpe'

print("Training Thai Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_th.txt',
    model_prefix='spm_th',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer (for Thai pair)...")
spm.SentencePieceTrainer.train(
    input='train_en_th.txt',
    model_prefix='spm_en_th',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Tokenizer training complete!")

Training Thai Tokenizer...
Training English Tokenizer (for Thai pair)...
Tokenizer training complete!


In [6]:
# Load the processors
sp_th = spm.SentencePieceProcessor(model_file='spm_th.model')
sp_en = spm.SentencePieceProcessor(model_file='spm_en_th.model')

# Test Tokenization
idx = 0
print(f"Original th: {df.iloc[idx]['th']}")
print(f"Tokens: {sp_th.encode(df.iloc[idx]['th'], out_type=str)}")
print(f"IDs: {sp_th.encode(df.iloc[idx]['th'], out_type=int)}")

Original th: อิตาลีได้เอาชนะโปรตุเกสด้วยคะแนน31ต่อ5 ในกลุ่มc ของการแข่งขันรักบี้เวิลด์คัพปี2007 ที่สนามปาร์กเดแพร็งส์ ที่กรุงปารีส ประเทศฝรั่งเศส
Tokens: ['▁', 'อิตาลี', 'ได้', 'เอาชนะ', 'โปร', 'ตุ', 'เก', 'ส', 'ด้วยคะแนน', '3', '1', 'ต่อ', '5', '▁ใน', 'กลุ่ม', 'c', '▁ของ', 'การแข่งขัน', 'รัก', 'บ', 'ี้', 'เ', 'วิ', 'ล', 'ด์', 'ค', 'ัพ', 'ปี', '200', '7', '▁ที่', 'สนาม', 'ป', 'าร์', 'ก', 'เด', 'แพร', '็ง', 'ส์', '▁ที่', 'กรุง', 'ป', 'าร', 'ี', 'ส', '▁ประเทศ', 'ฝรั่งเศส']
IDs: [3866, 2645, 25, 2150, 2037, 170, 70, 3882, 2998, 3950, 3931, 120, 3947, 109, 321, 3929, 420, 806, 358, 3886, 59, 3872, 102, 3879, 389, 3888, 1481, 108, 2812, 3970, 237, 955, 3890, 374, 3869, 111, 1251, 1222, 729, 237, 1028, 3890, 4, 3877, 3882, 990, 1232]


## 4. PyTorch Dataset and DataLoader

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['th']
        trg_text = self.data.iloc[idx]['en']

        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]

        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)

    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)

    return src_pad, trg_pad

# Split Data
train_df = df.sample(frac=0.8, random_state=SEED)
val_test_df = df.drop(train_df.index)
val_df = val_test_df.sample(frac=0.5, random_state=SEED)
test_df = val_test_df.drop(val_df.index)

train_dataset = TranslationDataset(train_df, sp_th, sp_en)
val_dataset = TranslationDataset(val_df, sp_th, sp_en)
test_dataset = TranslationDataset(test_df, sp_th, sp_en)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 5. Transformer Model

In [8]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 d_model=512, nhead=8, num_encoder_layers=3,
                 num_decoder_layers=3, dim_feedforward=2048, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()

        self.d_model = d_model
        self.pad_idx = pad_idx

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.fc_out = nn.Linear(d_model, trg_vocab_size)
        self.init_weights()

    def init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, trg):
        src_key_padding_mask = (src == self.pad_idx)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)

        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))

        output = self.transformer(
            src=src_emb,
            tgt=trg_emb,
            tgt_mask=trg_mask,
            src_key_padding_mask=src_key_padding_mask
        )
        return self.fc_out(output)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

## 6. Training

In [9]:
SRC_VOCAB_SIZE = vocab_size
TRG_VOCAB_SIZE = vocab_size
D_MODEL = 256
N_HEAD = 4
NUM_LAYERS = 2
FF_DIM = 512
DROPOUT = 0.4
LR = 0.0005
EPOCHS = 100

model = TransformerModel(SRC_VOCAB_SIZE, TRG_VOCAB_SIZE, D_MODEL, N_HEAD, NUM_LAYERS, NUM_LAYERS, FF_DIM, DROPOUT).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

print("Starting training...")
for epoch in range(EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion, 1.0)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')
    # Save every epoch or best validation (skipped val loop for brevity here, but included in full code)
    torch.save(model.state_dict(), 'transformer_model_th.pt')

Starting training...
Epoch: 01 | Train Loss: 6.842
Epoch: 02 | Train Loss: 6.028
Epoch: 03 | Train Loss: 5.618
Epoch: 04 | Train Loss: 5.384
Epoch: 05 | Train Loss: 5.247
Epoch: 06 | Train Loss: 5.146
Epoch: 07 | Train Loss: 5.064
Epoch: 08 | Train Loss: 4.994
Epoch: 09 | Train Loss: 4.929
Epoch: 10 | Train Loss: 4.868
Epoch: 11 | Train Loss: 4.809
Epoch: 12 | Train Loss: 4.755
Epoch: 13 | Train Loss: 4.703
Epoch: 14 | Train Loss: 4.653
Epoch: 15 | Train Loss: 4.607
Epoch: 16 | Train Loss: 4.563
Epoch: 17 | Train Loss: 4.523
Epoch: 18 | Train Loss: 4.487
Epoch: 19 | Train Loss: 4.450
Epoch: 20 | Train Loss: 4.416
Epoch: 21 | Train Loss: 4.383
Epoch: 22 | Train Loss: 4.353
Epoch: 23 | Train Loss: 4.327
Epoch: 24 | Train Loss: 4.300
Epoch: 25 | Train Loss: 4.277
Epoch: 26 | Train Loss: 4.252
Epoch: 27 | Train Loss: 4.229
Epoch: 28 | Train Loss: 4.208
Epoch: 29 | Train Loss: 4.189
Epoch: 30 | Train Loss: 4.169
Epoch: 31 | Train Loss: 4.150
Epoch: 32 | Train Loss: 4.132
Epoch: 33 | Train L

In [10]:
# Save artifacts for Web App
import shutil
os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model_th.pt', 'app/models/transformer_model_th.pt')
shutil.copy('spm_th.model', 'app/models/spm_th.model')
shutil.copy('spm_en_th.model', 'app/models/spm_en_th.model')
print("Models copied to app/models/")

Models copied to app/models/
